# Data Access

In [1]:
from dotenv import load_dotenv
import os
import duckdb

In [16]:
# checking for credential presence
secrets = ["R2_SECRET_ACCESS_KEY", "R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_BUCKET_NAME"]
print("---Presence of Keys---")
for secret in secrets:
    print(f"{secret}: {os.getenv(secret) is not None}")

---Presence of Keys---
R2_SECRET_ACCESS_KEY: True
R2_ACCOUNT_ID: True
R2_ACCESS_KEY_ID: True
R2_BUCKET_NAME: True


In [ ]:
con = duckdb.connect()
con.sql("INSTALL httpfs;LOAD httpfs;") # allows to read files over the network instead of on my local disk

In [ ]:
# Registering r2 credentials with duckDB
con.sql(f"""
CREATE OR REPLACE SECRET r2_secret (
    TYPE S3,
    KEY_ID '{os.getenv("R2_ACCESS_KEY_ID")}',
    SECRET '{os.getenv("R2_SECRET_ACCESS_KEY")}',
    ENDPOINT '{os.getenv("R2_ACCOUNT_ID")}.r2.cloudflarestorage.com',
    URL_STYLE 'path',
    REGION 'auto'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [ ]:
# Confirm presence of credentials in DuckDB for usage in R2
con.sql("SELECT name, type FROM duckdb_secrets();")

┌───────────┬─────────┐
│   name    │  type   │
│  varchar  │ varchar │
├───────────┼─────────┤
│ r2_secret │ s3      │
└───────────┴─────────┘

# Initial Data Exploration

In [6]:
# checking for access to files
con.sql("""
SELECT * FROM glob('s3://crypto-data/parquet_staging/bbo/**/*.parquet')
LIMIT 20
"""
)

┌─────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                                                  file                                                   │
│                                                 varchar                                                 │
├─────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_05.parquet │
│ s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_06.parquet │
│ s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_07.parquet │
│ s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_08.parquet │
│ s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_09.parquet │
│ s3://crypto-data/parquet_s

In [ ]:
# random bbo information on 9th april at binance
df = con.sql("""
SELECT *
FROM read_parquet('s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/binance_bbo_2026-04-09_05.parquet')
LIMIT 5
""").pl()
df


timestamp,exchange,symbol,bid_price,bid_size,ask_price,ask_size,local_timestamp,date
"datetime[μs, Asia/Singapore]",str,str,f64,f64,f64,f64,i64,date
2026-04-09 13:00:00.013 +08,"""binance""","""BTC-USD-PERP""",70760.0,21.519,70760.1,6.491,null,2026-04-09
2026-04-09 13:00:00.026 +08,"""binance""","""BTC-USD-PERP""",70760.0,21.519,70760.1,6.503,null,2026-04-09
2026-04-09 13:00:00.080 +08,"""binance""","""BTC-USD-PERP""",70760.0,21.519,70760.1,6.505,null,2026-04-09
2026-04-09 13:00:00.112 +08,"""binance""","""BTC-USD-PERP""",70760.0,21.517,70760.1,6.505,null,2026-04-09
2026-04-09 13:00:00.134 +08,"""binance""","""BTC-USD-PERP""",70760.0,21.239,70760.1,6.505,null,2026-04-09


Observations from bbo information:
| Column Name | Type | Example | Description |
|------------|------|---------|-------------|
| `timestamp` | datetime (μs, Asia/Singapore) | `2026-04-09 13:00:00.013 +08` | Timestamp in Singapore timezone with millisecond/microsecond precision |
| `exchange` | String | `binance` | Name of the cryptocurrency exchange |
| `symbol` | String | `BTC-USD-PERP` | Trading instrument symbol (Bitcoin USD perpetual futures) |
| `bid_price` | Float64 | `70760.0` | Highest buying price currently available |
| `bid_size` | Float64 | `21.519` | Quantity available at the highest bid price |
| `ask_price` | Float64 | `70760.1` | Lowest selling price currently available |
| `ask_size` | Float64 | `6.505` | Quantity available at the lowest ask price |
| `local_timestamp` | i64 | `` | Check for nullity in data |
| `date` | Date | `2026-04-09` | Verify relation to timestamp |

In [20]:
# Checking for other symbols in the dataset
con.sql("""
SELECT DISTINCT symbol
FROM read_parquet('s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/*.parquet')
""").pl()

symbol
str
"""ETH-USD-SPOT"""
"""BTC-USD-PERP"""
"""SOL-USD-PERP"""
"""BTC-USD-SPOT"""
"""ETH-USD-PERP"""
"""SOL-USD-SPOT"""


Elucidation: There are SPOT and PERP for BTC, SOL, ETH

In [9]:
# check for nullity of local_timestamp
con.sql("""
SELECT 
    count(*) as total,
    count(local_timestamp) as non_null_local
FROM read_parquet('s3://crypto-data/parquet_staging/bbo/date=2026-04-09/exchange=binance/*.parquet')
""").pl()

total,non_null_local
i64,i64
82382834,0
